# Heatmaps and Clustering for RNA-seq (R)

_BioNexus Hub - bionexushub.com · Companion to the lesson._

**Set the runtime to R first:** Runtime, then Change runtime type, then R.


## 1. Set up and transform


In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install("DESeq2", update = FALSE, ask = FALSE)
install.packages("pheatmap")
library(DESeq2); library(pheatmap)


In [ ]:
base <- "https://raw.githubusercontent.com/owkin/PyDESeq2/main/datasets/synthetic/"
counts <- as.matrix(read.csv(paste0(base,"test_counts.csv"), row.names=1))
metadata <- read.csv(paste0(base,"test_metadata.csv"), row.names=1)
metadata$condition <- factor(metadata$condition)
metadata <- metadata[colnames(counts), , drop=FALSE]
dds <- DESeqDataSetFromMatrix(counts, metadata, design = ~ condition)
dds <- DESeq(dds)
res <- results(dds, contrast=c("condition","B","A"))
vsd <- vst(dds, blind=FALSE)   # use rlog(dds) on very small data


## 2. Sample-distance heatmap (QC)
Do replicates of the same condition cluster together?


In [ ]:
sampleDists <- dist(t(assay(vsd)))
pheatmap(as.matrix(sampleDists),
         clustering_distance_rows = sampleDists,
         clustering_distance_cols = sampleDists,
         main = "Sample-to-sample distances")


## 3. PCA
The same question, one glance.


In [ ]:
plotPCA(vsd, intgroup = "condition")


## 4. Gene heatmap of the top hits
Z-score each gene so the pattern shows, not the absolute level.


In [ ]:
top <- head(order(res$padj), 30)
mat <- assay(vsd)[top, ]
mat <- t(scale(t(mat)))
ann <- as.data.frame(colData(vsd)[, "condition", drop=FALSE])
pheatmap(mat, annotation_col = ann, show_rownames = FALSE,
         main = "Top 30 genes (z-scored)")


### Level up
Explore pheatmap options (palettes, cutree_rows) or the ComplexHeatmap package for publication figures.
